# Step 9: fe-bar-fraud-triage app validation

Workspace fevm-workshop-guy, 2026-11-04, conversation 1085628541759563.

App: https://fe-bar-fraud-triage-7474659392412718.aws.databricksapps.com
Service principal `app-1xzuu6 fe-bar-fraud-triage` (ID 72319122930742, application ID / Postgres role `8f239f44-078a-4f8f-90f1-4c1b0bcdc26c`). Active deployment `01f1d98ebd6c1954bd2d4b7f9d7c6ae5` (03:04 UTC 2026-11-06).

Run order: D.0 setup, D.1, D.2, D.3 (then STOP for the browser test), D.6, then D.4 after the browser test.

Browser test (2026-11-06 UTC): TXN-0045116 Explain + Open Case (00:38, repeated after the 03:04 redeploy), Ask Genie 03:07. The live before snapshot was taken at 22:39 UTC on 2026-11-05 (227 cases, 9 open auto_block, all inserted by alex.owen) and later overwritten by a re-run of D.3, so D.4 derives the before state from the data (cases created before the app's case) and gets the same 227 / 9.

D.5 (app request log): the app's /logz endpoint needs interactive OAuth and can't be read from a notebook, so request evidence comes from the audit log and query history and is written to `logs/app_requests_2026-11-04.md`.

In [0]:
# Same pinned packages the app installs (app/requirements.txt), so the app code path runs here unchanged.
%pip install -r /Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/app/requirements.txt --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import hashlib
from pathlib import Path

REPO = Path("/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo")
APP_SP = "8f239f44-078a-4f8f-90f1-4c1b0bcdc26c"
RESULTS = {}

def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

print(f"{'module':<24} {'repo root sha256':<66} {'app/ sha256':<66} result")
for m in ["fraud_triage_agent.py", "lakebase_cases.py"]:
    a, b = sha(REPO / m), sha(REPO / "app" / m)
    ok = a == b
    RESULTS[f"D.1 {m} byte-identical"] = ok
    print(f"{m:<24} {a}  {b}  {'PASS' if ok else 'FAIL'}")

module                   repo root sha256                                                   app/ sha256                                                        result
fraud_triage_agent.py    7ef49d2db69b8aabf082249b1c60065c0afc58dc9c3a41c66b15a174b2f74293  7ef49d2db69b8aabf082249b1c60065c0afc58dc9c3a41c66b15a174b2f74293  PASS
lakebase_cases.py        84364c4395330de0edf265459b6f18a697b5ef067c2396933cacec391e7aca40  84364c4395330de0edf265459b6f18a697b5ef067c2396933cacec391e7aca40  PASS


In [0]:
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
app = w.apps.get("fe-bar-fraud-triage")
print("app status:", app.app_status.state.value, "| deployment:", app.active_deployment.deployment_id, app.active_deployment.status.state.value)
for path in ["/", "/_stcore/health"]:
    r = requests.get(app.url + path, headers=w.config.authenticate(), timeout=20, allow_redirects=False)
    loc = r.headers.get("location", "")
    print(f"GET {path}: HTTP {r.status_code} -> {loc.split('?')[0]}")
print("\nResult: the app proxy redirects workspace-token requests to the interactive OAuth authorize page.")
print("Authenticating to the app from a notebook is not possible this way; not worked around. Run proof = browser test (D.3).")

app status: RUNNING | deployment: 01f1d98ebd6c1954bd2d4b7f9d7c6ae5 SUCCEEDED
GET /: HTTP 302 -> https://fevm-workshop-guy.cloud.databricks.com/oidc/oauth2/v2.0/authorize
GET /_stcore/health: HTTP 302 -> https://fevm-workshop-guy.cloud.databricks.com/oidc/oauth2/v2.0/authorize

Result: the app proxy redirects workspace-token requests to the interactive OAuth authorize page.
Authenticating to the app from a notebook is not possible this way; not worked around. Run proof = browser test (D.3).


In [0]:
# Run this BEFORE the browser test: it records the before state and picks the target.
# Re-running it after the test shows the post-test state and a new target (the original target now has a case);
# D.4 derives the true before state from the data, so these checks don't depend on this cell's latest output.
import sys, json
sys.path.insert(0, str(REPO))
from lakebase_cases import LakebaseCases

lead = LakebaseCases(pg_role="fraud_lead")
BEFORE = {
    "open_by_queue": lead.query("SELECT assigned_queue, count(*) AS n FROM cases.fraud_cases WHERE status='open' GROUP BY 1 ORDER BY 1"),
    "created_today": lead.query("SELECT count(*) AS n FROM cases.fraud_cases WHERE created_at::date = CURRENT_DATE")[0]["n"],
    "db_inserted_by": lead.query("SELECT db_inserted_by, count(*) AS n FROM cases.fraud_cases GROUP BY 1 ORDER BY 1"),
    "total": lead.query("SELECT count(*) AS n FROM cases.fraud_cases")[0]["n"],
}
print("STATE AT RUN TIME (", lead.query("SELECT now() AS t")[0]["t"], ") - the before state only when run ahead of the browser test")
for k, v in BEFORE.items():
    print(f"  {k}: {v}")

# Target: highest-risk block transaction with no case and not high amount (<= 10000)
TARGET = lead.query("""
  SELECT r.txn_id, r.risk_score::float AS risk_score, f.amount::float AS amount
  FROM fraud_gov.txn_risk_current r JOIN fraud_gov.txn_features f ON f.txn_id = r.txn_id
  WHERE r.recommended_action = 'block' AND f.amount <= 10000
    AND NOT EXISTS (SELECT 1 FROM cases.fraud_cases c WHERE c.txn_id = r.txn_id)
  ORDER BY r.risk_score DESC LIMIT 1""")[0]
TARGET_TXN = TARGET["txn_id"]
print(f"\nBrowser-test target: {TARGET_TXN}  risk_score={TARGET['risk_score']:.4f}  (amount <= 10000, no existing case)")
print("Cases for target now:", lead.query("SELECT count(*) AS n FROM cases.fraud_cases WHERE txn_id=%s", (TARGET_TXN,))[0]["n"])
lead.close()

STATE AT RUN TIME ( 2026-11-06 03:33:48.641422+00:00 ) - the before state only when run ahead of the browser test
  open_by_queue: [{'assigned_queue': 'auto_block', 'n': 10}]
  created_today: 1
  db_inserted_by: [{'db_inserted_by': '8f239f44-078a-4f8f-90f1-4c1b0bcdc26c', 'n': 1}, {'db_inserted_by': 'alex.owen@databricks.com', 'n': 227}]
  total: 228

Browser-test target: TXN-0061730  risk_score=0.9922  (amount <= 10000, no existing case)
Cases for target now: 0


In [0]:
import importlib.util, json

def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

# Load the app/ copies explicitly so the tools' `from lakebase_cases import ...` resolves to app/lakebase_cases.py
app_lc = _load("lakebase_cases", str(REPO / "app" / "lakebase_cases.py"))
app_agent = _load("fraud_triage_agent", str(REPO / "app" / "fraud_triage_agent.py"))
print("lakebase_cases from:", app_lc.__file__)
print("fraud_triage_agent from:", app_agent.__file__)

lead = app_lc.LakebaseCases(pg_role="fraud_lead")
total_before = lead.query("SELECT count(*) AS n FROM cases.fraud_cases")[0]["n"]

# 1. Unknown transaction is rejected
r1 = json.loads(app_agent.create_fraud_case.invoke({"txn_id": "TXN-DOES-NOT-EXIST", "action": "block", "notes": "D.6 guardrail test"}))
ok1 = "error" in r1 and "case_id" not in r1
print("\nunknown txn ->", r1, "|", "PASS" if ok1 else "FAIL")

# 2. Approve-band transaction pushed to block is refused
appr = lead.query("SELECT txn_id, risk_score::float AS s FROM fraud_gov.txn_risk_current WHERE recommended_action='approve' ORDER BY risk_score DESC LIMIT 1")[0]
r2 = json.loads(app_agent.create_fraud_case.invoke({"txn_id": appr["txn_id"], "action": "block", "notes": "D.6 guardrail test"}))
ok2 = "error" in r2 and "case_id" not in r2
print(f"approve-band {appr['txn_id']} (score {appr['s']:.4f}) as block ->", r2, "|", "PASS" if ok2 else "FAIL")

total_after = lead.query("SELECT count(*) AS n FROM cases.fraud_cases")[0]["n"]
ok3 = total_after == total_before
print(f"\ncases before={total_before} after={total_after} -> no case written: {'PASS' if ok3 else 'FAIL'}")
RESULTS.update({"D.6 unknown txn rejected": ok1, "D.6 approve-band block refused": ok2, "D.6 no case written": ok3})
lead.close()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-5f0e8b18-0a63-401a-8142-aafc56ccdbe3/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026/11/06 03:30:58 WARNING mlflow.utils.autologging_utils: MLflow langchain autologging is known to be compatible with 1.0.4 <= langchain, but the installed version is 0.3.30. If you encounter errors during autologging, try upgrading / downgrading langchain to a compatible version, or try upgrading MLflow.


lakebase_cases from: /Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/app/lakebase_cases.py
fraud_triage_agent from: /Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/app/fraud_triage_agent.py


2026/11/06 03:31:00 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.



unknown txn -> {'error': "txn_id 'TXN-DOES-NOT-EXIST' not found in fraud.txn_features"} | PASS
approve-band TXN-0278800 (score 0.8714) as block -> {'error': "Refused: model recommended 'approve' for TXN-0278800; an approve-band transaction cannot be opened as 'block'."} | PASS

cases before=228 after=228 -> no case written: PASS


[Trace(trace_id='tr-4626010f622410d4e841e9ac94e8df88'), Trace(trace_id='tr-f5ad79c5224a53d687c405b784c0ba5e')]

In [0]:
# Case shown by the app after the browser test (Open Case clicked at 00:38 UTC and again after redeploy):
CASE_ID_FROM_APP = "5af3575c-cda8-47cb-97be-021b26dca000"
TARGET_TXN = "TXN-0045116"

import sys
sys.path.insert(0, str(REPO))
from lakebase_cases import LakebaseCases
lead = LakebaseCases(pg_role="fraud_lead")
# D.3's live snapshot was taken before the D.6 artifact case (378c7c37) was deleted and was not re-run
# before the first click, so derive the true before state from the data: every case created before the app's case.
new_ts = lead.query("SELECT created_at FROM cases.fraud_cases WHERE case_id::text=%s", (CASE_ID_FROM_APP,))[0]["created_at"]
BEFORE = {
    "open_by_queue": lead.query("SELECT assigned_queue, count(*) AS n FROM cases.fraud_cases WHERE status='open' AND created_at < %s GROUP BY 1 ORDER BY 1", (new_ts,)),
    "db_inserted_by": lead.query("SELECT db_inserted_by, count(*) AS n FROM cases.fraud_cases WHERE created_at < %s GROUP BY 1 ORDER BY 1", (new_ts,)),
    "total": lead.query("SELECT count(*) AS n FROM cases.fraud_cases WHERE created_at < %s", (new_ts,))[0]["n"],
}
print("BEFORE (derived: created_at <", new_ts, ")")
for k, v in BEFORE.items():
    print(f"  {k}: {v}")
AFTER = {
    "open_by_queue": lead.query("SELECT assigned_queue, count(*) AS n FROM cases.fraud_cases WHERE status='open' GROUP BY 1 ORDER BY 1"),
    "created_today": lead.query("SELECT count(*) AS n FROM cases.fraud_cases WHERE created_at::date = CURRENT_DATE")[0]["n"],
    "db_inserted_by": lead.query("SELECT db_inserted_by, count(*) AS n FROM cases.fraud_cases GROUP BY 1 ORDER BY 1"),
    "total": lead.query("SELECT count(*) AS n FROM cases.fraud_cases")[0]["n"],
}
print("AFTER")
for k, v in AFTER.items():
    print(f"  {k}: {v}")

row = lead.query("""SELECT case_id::text AS case_id, txn_id, status, assigned_queue, created_by, db_inserted_by, created_at
                    FROM cases.fraud_cases WHERE case_id::text = %s""", (CASE_ID_FROM_APP,))
print("\nNew case row:", row)
n_for_txn = lead.query("SELECT count(*) AS n FROM cases.fraud_cases WHERE txn_id=%s", (TARGET_TXN,))[0]["n"]
lead.close()

c = row[0] if row else {}
RESULTS["D.4 case exists and is for target txn"] = bool(row) and c["txn_id"] == TARGET_TXN
RESULTS["D.4 db_inserted_by is app Postgres role"] = c.get("db_inserted_by") == APP_SP
RESULTS["D.4 not attributed to alex.owen"] = c.get("db_inserted_by") != "alex.owen@databricks.com"
RESULTS["D.4 exactly one case for txn after two clicks"] = n_for_txn == 1
RESULTS["D.4 total grew by exactly 1"] = AFTER["total"] == BEFORE["total"] + 1
_l = LakebaseCases(pg_role="fraud_lead")
n_actions = _l.query("SELECT count(*) AS n FROM cases.case_actions WHERE case_id::text=%s", (CASE_ID_FROM_APP,))[0]["n"]
_l.close()
print("case_actions rows for case:", n_actions)
RESULTS["D.4 repeat click added no case_actions"] = n_actions == 1

# Visible through UC with no sync/job in between (fraud_dash is a federated view over Lakebase)
uc = spark.sql(f"SELECT * FROM fe_bar_lakebase.fraud_dash.case_summary WHERE case_id = '{CASE_ID_FROM_APP}'").collect()
print("\nUC fe_bar_lakebase.fraud_dash.case_summary rows:", [r.asDict() for r in uc])
RESULTS["D.4 case visible in UC case_summary"] = len(uc) == 1

for k, v in RESULTS.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")

BEFORE (derived: created_at < 2026-11-06 00:38:31.073759+00:00 )
  open_by_queue: [{'assigned_queue': 'auto_block', 'n': 9}]
  db_inserted_by: [{'db_inserted_by': 'alex.owen@databricks.com', 'n': 227}]
  total: 227
AFTER
  open_by_queue: [{'assigned_queue': 'auto_block', 'n': 10}]
  created_today: 1
  db_inserted_by: [{'db_inserted_by': '8f239f44-078a-4f8f-90f1-4c1b0bcdc26c', 'n': 1}, {'db_inserted_by': 'alex.owen@databricks.com', 'n': 227}]
  total: 228

New case row: [{'case_id': '5af3575c-cda8-47cb-97be-021b26dca000', 'txn_id': 'TXN-0045116', 'status': 'open', 'assigned_queue': 'auto_block', 'created_by': 'fraud_triage_copilot:agent', 'db_inserted_by': '8f239f44-078a-4f8f-90f1-4c1b0bcdc26c', 'created_at': datetime.datetime(2026, 11, 6, 0, 38, 31, 73759, tzinfo=zoneinfo.ZoneInfo(key='GMT'))}]
case_actions rows for case: 1

UC fe_bar_lakebase.fraud_dash.case_summary rows: [{'case_id': '5af3575c-cda8-47cb-97be-021b26dca000', 'txn_id': 'TXN-0045116', 'customer_id': 'CUST-05980', 'status

In [0]:
# The app's Ask Genie call is recovered from platform records (app /logz is OAuth-only and unreadable here):
#   conversation id  -> system.access.audit (aibiGenie events by the app SP)
#   generated SQL    -> query history (statements by the app SP tagged with the Genie space)
APP_SP_ID, SPACE = 72319122930742, "01f1d7b464d510b9aa6cde48300787ca"

aud = spark.sql(f"""
  SELECT request_params['conversation_id'] AS conversation_id, request_params['message_id'] AS message_id,
         min(event_time) AS first_t, collect_set(action_name) AS actions, max(response.status_code) AS max_status
  FROM system.access.audit
  WHERE event_date >= '2026-11-06' AND service_name = 'aibiGenie' AND event_time >= '2026-11-06T03:04:00'
    AND user_identity.email = '{APP_SP}' AND request_params['space_id'] = '{SPACE}'
    AND request_params['conversation_id'] IS NOT NULL
  GROUP BY 1, 2 ORDER BY first_t""").collect()
print("Genie audit (app SP):", [r.asDict() for r in aud])
GENIE_CONVERSATION_ID = aud[0]["conversation_id"] if aud else None

from databricks.sdk import WorkspaceClient
h = WorkspaceClient().api_client.do("GET", "/api/2.0/sql/history/queries", query={
    "filter_by.user_ids": APP_SP_ID, "filter_by.query_start_time_range.start_time_ms": 1793934240000, "max_results": 50})
genie_q = [q for q in h.get("res", []) if (q.get("query_source") or {}).get("genie_space_id") == SPACE
           and not q["query_text"].startswith("DESCRIBE")]
GENIE_SQL = genie_q[-1]["query_text"] if genie_q else None
print("Genie conversation:", GENIE_CONVERSATION_ID)
print("Generated SQL (run as", genie_q[-1]["user_name"] if genie_q else None, "):\n", GENIE_SQL)

genie_rows = {r["strategy"]: float(r["fraud_amount_caught"]) for r in spark.sql(GENIE_SQL).collect()}
bc = {r["strategy"]: float(r["fraud_amount_caught"]) for r in spark.sql("SELECT * FROM fe_bar_lakebase.fraud_dash.baseline_comparison").collect()}
print("Genie SQL result:", genie_rows, "| baseline_comparison:", bc)
RESULTS["D.4 Genie conversation by app SP found"] = GENIE_CONVERSATION_ID is not None
RESULTS["D.4 Genie SQL executed as app SP"] = bool(genie_q) and genie_q[-1]["user_name"] == APP_SP
RESULTS["D.4 Genie numbers match baseline_comparison"] = genie_rows == {k: bc[k] for k in genie_rows} and set(genie_rows) == {"model", "baseline_rule"}
for k, v in RESULTS.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")

Genie audit (app SP): [{'conversation_id': '01f1d98f28021142877f6ea599ede8a1', 'message_id': '01f1d98f281b1feb86a66e6847689b8b', 'first_t': datetime.datetime(2026, 11, 6, 3, 7, 10, 356000), 'actions': ['genieGetConversationMessage'], 'max_status': 200}]
Genie conversation: 01f1d98f28021142877f6ea599ede8a1
Generated SQL (run as 8f239f44-078a-4f8f-90f1-4c1b0bcdc26c ):
 SELECT `strategy`, `rule_text`, `fraud_amount_caught`, `total_fraud_amount` FROM `fe_bar_lakebase`.`fraud_dash`.`baseline_comparison` WHERE `strategy` IN ('model', 'baseline_rule') ORDER BY `fraud_amount_caught` DESC
Genie SQL result: {'model': 7010482.03, 'baseline_rule': 933889.13} | baseline_comparison: {'baseline_rule': 933889.13, 'model': 7010482.03}
PASS  D.1 fraud_triage_agent.py byte-identical
PASS  D.1 lakebase_cases.py byte-identical
PASS  D.6 unknown txn rejected
PASS  D.6 approve-band block refused
PASS  D.6 no case written
PASS  D.4 case exists and is for target txn
PASS  D.4 db_inserted_by is app Postgres rol